# Pythonic Data Cleaning With Pandas and NumPy

In [1]:
print('hello')

hello


In [1]:
import pandas as pd
import numpy as np
import datetime as dt

### Dropping Columns in a DataFrame

In [2]:
df = pd.read_csv('data/BL-Flickr-Images-Book.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/BL-Flickr-Images-Book.csv'

Use `drop()` to get rid of unwanted columns:

In [ ]:
to_drop = ['Edition Statement',
           'Corporate Author',
           'Corporate Contributors',
           'Former owner',
           'Engraver',
           'Contributors',
           'Issuance type',
           'Shelfmarks']

df.drop(to_drop, inplace=True, axis=1)

In [ ]:
df.head()

,Identifier,Place of Publication,Date of Publication,Publisher,Title,Author,Flickr URL
0,206,London,1879 [1878],S. Tinsley & Co.,Walter Forbes. [A novel.] By A. A,A. A.,http://www.flickr.com/photos/britishlibrary/ta...
1,216,London; Virtue & Yorston,1868,Virtue & Co.,All for Greed. [A novel. The dedication signed...,"A., A. A.",http://www.flickr.com/photos/britishlibrary/ta...
2,218,London,1869,"Bradbury, Evans & Co.",Love the Avenger. By the author of “All for Gr...,"A., A. A.",http://www.flickr.com/photos/britishlibrary/ta...
3,472,London,1851,James Darling,"Welsh Sketches, chiefly ecclesiastical, to the...","A., E. S.",http://www.flickr.com/photos/britishlibrary/ta...
4,480,London,1857,Wertheim & Macintosh,"[The World in which I live, and my place in it...","A., E. S.",http://www.flickr.com/photos/britishlibrary/ta...


A more readable and intuitive alternative with `columns`:

<code> columns=to_drop, inplace=True </code>

If you already know which columns you want to keep, `usecols` is also a good choice.

### Changing the Index of a DataFrame

In [ ]:
df['Identifier'].is_unique

True

In [ ]:
df = df.set_index('Identifier')
df.head()

,Place of Publication,Date of Publication,Publisher,Title,Author,Flickr URL
Identifier,,,,,,
206,London,1879 [1878],S. Tinsley & Co.,Walter Forbes. [A novel.] By A. A,A. A.,http://www.flickr.com/photos/britishlibrary/ta...
216,London; Virtue & Yorston,1868,Virtue & Co.,All for Greed. [A novel. The dedication signed...,"A., A. A.",http://www.flickr.com/photos/britishlibrary/ta...
218,London,1869,"Bradbury, Evans & Co.",Love the Avenger. By the author of “All for Gr...,"A., A. A.",http://www.flickr.com/photos/britishlibrary/ta...
472,London,1851,James Darling,"Welsh Sketches, chiefly ecclesiastical, to the...","A., E. S.",http://www.flickr.com/photos/britishlibrary/ta...
480,London,1857,Wertheim & Macintosh,"[The World in which I live, and my place in it...","A., E. S.",http://www.flickr.com/photos/britishlibrary/ta...


Accessing index by label with `loc[]`:

In [ ]:
df.loc[206]

Place of Publication                                               London
Date of Publication                                           1879 [1878]
Publisher                                                S. Tinsley & Co.
Title                                   Walter Forbes. [A novel.] By A. A
Author                                                              A. A.
Flickr URL              http://www.flickr.com/photos/britishlibrary/ta...
Name: 206, dtype: object

Position-based search with `iloc[]`:

In [ ]:
df.iloc[206]

Place of Publication                                               London
Date of Publication                                                  1848
Publisher                                                 Richard Bentley
Title                   Rambles in the romantic regions of the Hartz M...
Author                                   Andersen, H. C. (Hans Christian)
Flickr URL              http://www.flickr.com/photos/britishlibrary/ta...
Name: 77554, dtype: object

### Tidying up Fields in the Data

In [ ]:
print(f" Number of objects: {len(df.dtypes)}") 

 Number of objects: 6


`'Date of Publication'` is a field where we can enforce a numeric value, as it is stored as object right now.

In [ ]:
df.loc[1905:, 'Date of Publication'].head(10)

Identifier
1905           1888
1929    1839, 38-54
2836           1897
2854           1865
2956        1860-63
2957           1873
3017           1866
3131           1899
4598           1814
4884           1820
Name: Date of Publication, dtype: object

However, there's a series of issues that we need to correct:
- Remove extra dates in square brackets
- Convert date ranges to their initial date
- Remove uncertain dates such as [1897?] and replace them with `NaN`
- Convert the string `nan` to NumPy's `NaN`

Let's use a single regular expression to extract the publication year:

<code> regex = r'^(\d{4})

Which is meant to find any four digits at the beginning of a string.
- r = regular expression
- \d = any digit
- {4} = repeats this rule four times
- ^ = matches the start of a string
- () = capturing group

In [ ]:
extr = df['Date of Publication'].str.extract(r'^(\d{4})', expand=False)
extr.head()

Identifier
206    1879
216    1868
218    1869
472    1851
480    1857
Name: Date of Publication, dtype: object

Get numerical version with `pd.to_numeric`:

In [ ]:
df['Date of Publication'] = pd.to_numeric(extr, downcast='integer')
df['Date of Publication'].dtype


dtype('float64')

In [ ]:
df['Date of Publication'].isnull().sum() / len(df)

0.11717147339205986

### Combining `str` Methods with NumPy to Clean Columns

Let's combine Pandas `str` with NumPy's `np.where` to clean the `Place of Publication` column. This will work as an Excel's `IF()`. This is the syntax:

<code> np.where(condition, then, else) </code>

In [ ]:
df['Place of Publication'].head(10)

Identifier
206                                  London
216                London; Virtue & Yorston
218                                  London
472                                  London
480                                  London
481                                  London
519                                  London
667     pp. 40. G. Bryan & Co: Oxford, 1898
874                                 London]
1143                                 London
Name: Place of Publication, dtype: object

Let's check a few rows:

In [ ]:
df.loc[4157862]

Place of Publication                                  Newcastle-upon-Tyne
Date of Publication                                                1867.0
Publisher                                                      T. Fordyce
Title                   Local Records; or, Historical Register of rema...
Author                      FORDYCE, T. - Printer, of Newcastle-upon-Tyne
Flickr URL              http://www.flickr.com/photos/britishlibrary/ta...
Name: 4157862, dtype: object

In [ ]:
df.loc[4159587]

Place of Publication                                  Newcastle upon Tyne
Date of Publication                                                1834.0
Publisher                                                Mackenzie & Dent
Title                   An historical, topographical and descriptive v...
Author                                              Mackenzie, E. (Eneas)
Flickr URL              http://www.flickr.com/photos/britishlibrary/ta...
Name: 4159587, dtype: object

As we can see, both books were published in the same time, yet the city names are different. Let's use `str.contains()` to get a Boolean mask.

In [ ]:
pub = df['Place of Publication']
london = pub.str.contains('London')
london[:5]

Identifier
206    True
216    True
218    True
472    True
480    True
Name: Place of Publication, dtype: bool

In [ ]:
oxford = pub.str.contains('Oxford')

Combining both with `np.where()`:

In [ ]:
df['Place of Publication'] = np.where(london, 'London',
                                      np.where(oxford, 'Oxford',
                                               pub.str.replace('-',' ')))

df['Place of Publication'].head()

Identifier
206    London
216    London
218    London
472    London
480    London
Name: Place of Publication, dtype: object

In [ ]:
df.head()

,Place of Publication,Date of Publication,Publisher,Title,Author,Flickr URL
Identifier,,,,,,
206,London,1879.0,S. Tinsley & Co.,Walter Forbes. [A novel.] By A. A,A. A.,http://www.flickr.com/photos/britishlibrary/ta...
216,London,1868.0,Virtue & Co.,All for Greed. [A novel. The dedication signed...,"A., A. A.",http://www.flickr.com/photos/britishlibrary/ta...
218,London,1869.0,"Bradbury, Evans & Co.",Love the Avenger. By the author of “All for Gr...,"A., A. A.",http://www.flickr.com/photos/britishlibrary/ta...
472,London,1851.0,James Darling,"Welsh Sketches, chiefly ecclesiastical, to the...","A., E. S.",http://www.flickr.com/photos/britishlibrary/ta...
480,London,1857.0,Wertheim & Macintosh,"[The World in which I live, and my place in it...","A., E. S.",http://www.flickr.com/photos/britishlibrary/ta...


In [ ]:
df['Place of Publication'] = df['Place of Publication'].astype('category')

In [ ]:
df.dtypes

Place of Publication    category
Date of Publication      float64
Publisher                 object
Title                     object
Author                    object
Flickr URL                object
dtype: object

### Cleaning the Entire Dataset Using the applymap Function

 Pandas `.applymap()` method is similar to the in-built `map()` function and simply applies a function to all the elements in a DataFrame. Let's use a different example now.

In [ ]:
university_towns = []
with open('data/university_towns.txt') as file:
    for line in file:
        if '[edit]' in line:
            # Remember this `state` until the next is found.
            state = line
        else:
            # Otherwise, we have a city; keep `state` as last seen
            university_towns.append((state, line))

university_towns[:5]

[('Alabama[edit]\n', 'Auburn (Auburn University)[1]\n'),
 ('Alabama[edit]\n', 'Florence (University of North Alabama)\n'),
 ('Alabama[edit]\n', 'Jacksonville (Jacksonville State University)[2]\n'),
 ('Alabama[edit]\n', 'Livingston (University of West Alabama)[2]\n'),
 ('Alabama[edit]\n', 'Montevallo (University of Montevallo)[2]\n')]

Let's wrap the list in a DataFrame and set columns as `State` and `RegionName`.

In [ ]:
towns_df = pd.DataFrame(university_towns,
                        columns=['State', 'RegionName'])
towns_df.head()

,State,RegionName
0,Alabama[edit]\n,Auburn (Auburn University)[1]\n
1,Alabama[edit]\n,Florence (University of North Alabama)\n
2,Alabama[edit]\n,Jacksonville (Jacksonville State University)[2]\n
3,Alabama[edit]\n,Livingston (University of West Alabama)[2]\n
4,Alabama[edit]\n,Montevallo (University of Montevallo)[2]\n


Use the `.applymap()` function to clean elements independently. But first, let's define a function that will do the job for us.

In [ ]:
def get_citystate(item):
    """Clean city and state names."""
    if ' (' in item:
        return item[:item.find(' (')]
    elif '[' in item:
        return item[:item.find('[')]
    else:
        return item

This function will check each DataFrame element individually. Then, it will look for a `(` or a `[` in each element.

In [ ]:
towns_df = towns_df.applymap(get_citystate)

After using `applymap()` on the object, let's see the results:

In [ ]:
towns_df.head()

,State,RegionName
0,Alabama,Auburn
1,Alabama,Florence
2,Alabama,Jacksonville
3,Alabama,Livingston
4,Alabama,Montevallo


That was great! Keep in mind though that when it comes to larger datasets, `applymap()` can take much longer!

### Renaming Columns and Skipping Rows


In [ ]:
olympics_df = pd.read_csv('data/olympics.csv')

In [ ]:
olympics_df.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,NaN,? Summer,01 !,02 !,03 !,Total,? Winter,01 !,02 !,03 !,Total,? Games,01 !,02 !,03 !,Combined total
1,Afghanistan (AFG),13,0,0,2,2,0,0,0,0,0,13,0,0,2,2
2,Algeria (ALG),12,5,2,8,15,3,0,0,0,0,15,5,2,8,15
3,Argentina (ARG),23,18,24,28,70,18,0,0,0,0,41,18,24,28,70
4,Armenia (ARM),5,1,2,9,12,6,0,0,0,0,11,1,2,9,12


What a messy dataset! Look, the headers are labeled as indexes, while the names of columns are at `row[0]`:

In [ ]:
olympics_df.iloc[0]

0                NaN
1           ? Summer
2               01 !
3               02 !
4               03 !
5              Total
6           ? Winter
7               01 !
8               02 !
9               03 !
10             Total
11           ? Games
12              01 !
13              02 !
14              03 !
15    Combined total
Name: 0, dtype: object

Also, the column headers are weird. Hence, we need to skip one row and set the header as the first row, while also renaming columns.

Skipping rows with `read_csv()` is easy:

In [ ]:
olympics_df = pd.read_csv('data/olympics.csv', header=1)

In [ ]:
olympics_df.head()

,Unnamed: 0,? Summer,01 !,02 !,03 !,Total,? Winter,01 !.1,02 !.1,03 !.1,Total.1,? Games,01 !.2,02 !.2,03 !.2,Combined total
0,Afghanistan (AFG),13,0,0,2,2,0,0,0,0,0,13,0,0,2,2
1,Algeria (ALG),12,5,2,8,15,3,0,0,0,0,15,5,2,8,15
2,Argentina (ARG),23,18,24,28,70,18,0,0,0,0,41,18,24,28,70
3,Armenia (ARM),5,1,2,9,12,6,0,0,0,0,11,1,2,9,12
4,Australasia (ANZ) [ANZ],2,3,4,5,12,0,0,0,0,0,2,3,4,5,12


Notice how the formerly `NaN` value in the countries column has changed to `Unnamed: 0`.

Renaming columns using `rename()`. First, we create a dictionary, where keys are column names to be replaced by the dict's values.

In [ ]:
new_names = {'Unnamed: 0' : 'Country',
             '? Summer': 'Summer Olympics',
             '01 !': 'Gold',
             '02 !': 'Silver',
             '03 !': 'Bronze',
             '? Winter': 'Winter Olympics',
             '01 !.1': 'Gold.1',
             '02 !.1': 'Silver.1',
             '03 !.1': 'Bronze.1',
             '? Games': '# Games',
             '01 !.2' : 'Gold.2',
             '02 !.2' : 'Silver.2',
             '03 !.2' : 'Bronze.2',
            }

In [ ]:
olympics_df.rename(columns=new_names, inplace=True)

In [ ]:
olympics_df.head()

,Country,Summer Olympics,Gold,Silver,Bronze,Total,Winter Olympics,Gold.1,Silver.1,Bronze.1,Total.1,# Games,Bronze.2,Silver.2,Bronze.2,Combined total
0,Afghanistan (AFG),13,0,0,2,2,0,0,0,0,0,13,0,0,2,2
1,Algeria (ALG),12,5,2,8,15,3,0,0,0,0,15,5,2,8,15
2,Argentina (ARG),23,18,24,28,70,18,0,0,0,0,41,18,24,28,70
3,Armenia (ARM),5,1,2,9,12,6,0,0,0,0,11,1,2,9,12
4,Australasia (ANZ) [ANZ],2,3,4,5,12,0,0,0,0,0,2,3,4,5,12


In [ ]:
import pandas as pd
import numpy as np

# Sample data
df = pd.DataFrame({
    'A': [1, np.nan, 3, np.nan, 5],
    'B': [np.nan, 'x', np.nan, 'y', 'z']
})

arr = np.array([10, np.nan, np.nan, 40, np.nan])

# ===== PANDAS BUILT-IN FUNCTIONS =====

# 1. fillna() – Fill NaNs with a specific value
df_filled = df.fillna(0)
print("1. fillna():\n", df_filled)

# 2. ffill() – Forward fill
df_ffill = df.ffill()
print("\n2. ffill():\n", df_ffill)

# 3. bfill() – Backward fill
df_bfill = df.bfill()
print("\n3. bfill():\n", df_bfill)

# 4. interpolate() – Fill numeric NaNs using interpolation
df_interp = df.interpolate()
print("\n4. interpolate():\n", df_interp)

# ===== NUMPY BUILT-IN FUNCTIONS =====

# Note: NumPy doesn't have built-in ffill/bfill, but we can use np.nan_to_num

# 5. nan_to_num() – Replace NaN with 0 or specified value
arr_filled = np.nan_to_num(arr, nan=0)
print("\n5. nan_to_num():\n", arr_filled)


And that's it for today! Thank you Real Python and thank you Malay for this awesome [tutorial](https://realpython.com/python-data-cleaning-numpy-pandas/)!